# ACME IM — Distribution Insights: Full Demo Notebook
**All 4 sections | Snowflake Notebooks native | No local auth required**

| Section | Topic |
|---------|-------|
| 1 | Salesforce Zero-Copy Integration |
| 2 | AI Pipeline Management (Snowpipe V2, Dynamic Tables) |
| 3 | Cortex AI Observability & Alerting |
| 4 | Self-Service Analytics via Cortex Analyst |

> **Run order**: Execute cells top to bottom. Warehouse: `INGEST`

In [ ]:
# Setup — runs natively in Snowflake Notebooks (no keypair auth needed)
from snowflake.snowpark.context import get_active_session
from snowflake.cortex import Complete
import json, pandas as pd

session = get_active_session()
session.sql('USE WAREHOUSE INGEST').collect()
session.sql('USE DATABASE ANALYTICS_DEV_DB').collect()
session.sql('USE SCHEMA ANALYTICS_DEV_DB.DISTRIBUTION').collect()

print(f'Account : {session.get_current_account()}')
print(f'Context : {session.get_current_database()}.{session.get_current_schema()}')


---
## Section 1: Salesforce Zero-Copy Integration
Salesforce data lands in Snowflake via Snowpipe V2 — no ETL, no data movement.
Distribution signals computed by live join of SFDC opportunities with dimension tables.

In [ ]:
-- Demo 1-A: Live Salesforce opportunities (Zero-Copy — no ETL)
SELECT opportunity_id, advisor_id, stage,
       ROUND(COALESCE(amount,0)/1e6, 2)    AS amount_m,
       close_date,
       row_timestamp                        AS last_updated
FROM   ANALYTICS_DEV_DB.STAGING.SFDC_OPPORTUNITY
ORDER  BY amount DESC NULLS LAST
LIMIT  10


In [ ]:
-- Demo 1-B: Distribution signals — SFDC joined with Advisor + Territory dims
SELECT a.advisor_name,
       t.territory_name,
       a.tier,
       COUNT(DISTINCT o.opportunity_id)       AS open_opps,
       ROUND(SUM(COALESCE(o.amount,0))/1e6,2) AS pipeline_m,
       CASE WHEN COUNT(DISTINCT o.opportunity_id) > 3 THEN 'HOT'
            WHEN COUNT(DISTINCT o.opportunity_id) > 1 THEN 'WARM'
            ELSE 'COLD' END                   AS signal
FROM   ANALYTICS_DEV_DB.STAGING.ADVISOR_DIM a
JOIN   ANALYTICS_DEV_DB.STAGING.TERRITORY_DIM t
       ON  a.territory_id = t.territory_id
LEFT JOIN ANALYTICS_DEV_DB.STAGING.SFDC_OPPORTUNITY o
       ON  a.advisor_id = o.advisor_id
       AND o.stage NOT IN ('Closed Won','Closed Lost')
GROUP  BY a.advisor_id, a.advisor_name, t.territory_name, a.tier
HAVING open_opps > 0
ORDER  BY pipeline_m DESC
LIMIT  15


In [ ]:
# Demo 1-C: Cortex AI brief from the top distribution signals
# s1_signals is the result of the SQL cell above (Snowflake Notebooks auto-binding)
top5 = s1_signals.to_pandas().head(5).to_dict(orient='records')  # noqa: F821
prompt = (
    'You are a distribution analytics assistant.\n'
    'Summarize these distribution signals in 3 bullets for a sales manager.\n'
    'Focus on HOT pipeline opportunities and one action recommendation.\n'
    f'Data: {json.dumps(top5, indent=2)}\nSummary:'
)
print('=== AI DISTRIBUTION BRIEF ===')
print(Complete('mistral-7b', prompt))


---
## Section 2: AI Pipeline Management
Snowpipe V2 streams advisor events in sub-second latency.
Three Dynamic Tables compute derived metrics incrementally:
- `ADVISOR_ENGAGEMENT_SCORE` (1 h lag)
- `FUND_FLOW_ATTRIBUTION` (1 d lag)
- `TERRITORY_HEAT_MAP` (4 h lag, reads staging directly to avoid cascading lag)

In [ ]:
-- Demo 2-A: Snowpipe V2 throughput and lag (last 24 hours)
SELECT MAX(row_timestamp)                                          AS latest_event,
       DATEDIFF('second', MAX(row_timestamp), CURRENT_TIMESTAMP()) AS seconds_lag,
       COUNT(*)                                                    AS total_events_24h,
       COUNT(DISTINCT advisor_id)                                  AS unique_advisors
FROM   ANALYTICS_DEV_DB.STAGING.ADVISOR_EVENTS_RAW
WHERE  row_timestamp >= DATEADD('hour', -24, CURRENT_TIMESTAMP())


In [ ]:
-- Demo 2-B: Dynamic Table — top advisor engagement scores (most recent)
SELECT advisor_name, territory_name, advisor_tier AS tier,
       engagement_score, aum_amount, open_opportunity_value,
       call_count_30d, meeting_count_30d
FROM   ANALYTICS_DEV_DB.DISTRIBUTION.ADVISOR_ENGAGEMENT_SCORE
WHERE  score_date = (SELECT MAX(score_date)
                     FROM ANALYTICS_DEV_DB.DISTRIBUTION.ADVISOR_ENGAGEMENT_SCORE)
ORDER  BY aum_amount DESC NULLS LAST
LIMIT  10


In [ ]:
-- Demo 2-C: Dynamic Table — fund flow attribution (this quarter)
SELECT fund_name, fund_category, flow_type,
       SUM(flow_amount)            AS net_flow,
       COUNT(DISTINCT advisor_id)  AS advisors
FROM   ANALYTICS_DEV_DB.DISTRIBUTION.FUND_FLOW_ATTRIBUTION
WHERE  flow_date >= DATE_TRUNC('quarter', CURRENT_DATE())
GROUP  BY fund_name, fund_category, flow_type
ORDER  BY ABS(net_flow) DESC
LIMIT  10


In [ ]:
# Demo 2-D: Cortex AI propensity scoring for attrition risk
scores_df = s2_engagement.to_pandas()  # noqa: F821

def classify_risk(row):
    prompt = (
        f'Classify attrition risk: HIGH, MEDIUM, or LOW. '
        f'Engagement: {row["ENGAGEMENT_SCORE"]:.0f}/100, '
        f'Calls 30d: {row["CALL_COUNT_30D"]}, '
        f'Meetings 30d: {row["MEETING_COUNT_30D"]}. '
        'Respond with one word only.'
    )
    try:
        return Complete('mistral-7b', prompt).strip().upper()[:6]
    except:
        return 'UNKNWN'

scores_df['ATTRITION_RISK'] = scores_df.head(5).apply(classify_risk, axis=1)
print('ADVISOR PROPENSITY SCORES:')
display(scores_df[['ADVISOR_NAME','ADVISOR_TIER','ENGAGEMENT_SCORE','ATTRITION_RISK']].head(5))  # noqa


---
## Section 3: Cortex AI Observability & Alerting
Data quality monitored by Snowflake DMFs (Data Metric Functions).
Alerts fire to Slack/email when engagement drops, flows spike, or pipeline stalls.

> `ALERT_HISTORY` has up to 2-hour lag. Run alerts first with `./manage.sh run-alerts`.

In [ ]:
-- Demo 3-A: DMF results (last 24 hours, STAGING + DISTRIBUTION only)
SELECT TABLE_NAME, METRIC_NAME, VALUE, MEASUREMENT_TIME,
       CASE WHEN (METRIC_NAME LIKE '%NULL%' OR METRIC_NAME LIKE '%DUPLICATE%')
                 AND VALUE > 0
            THEN 'WARNING' ELSE 'OK' END AS health_status
FROM   SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS
WHERE  TABLE_DATABASE = 'ANALYTICS_DEV_DB'
  AND  TABLE_SCHEMA IN ('STAGING', 'DISTRIBUTION')
  AND  MEASUREMENT_TIME >= DATEADD('hour', -24, CURRENT_TIMESTAMP())
ORDER  BY MEASUREMENT_TIME DESC


In [ ]:
-- Demo 3-B: Alert history (last 6 hours)
-- Correct columns: NAME, SCHEMA_NAME, SCHEDULED_TIME, STATE
SELECT NAME, SCHEMA_NAME, SCHEDULED_TIME, STATE
FROM   SNOWFLAKE.ACCOUNT_USAGE.ALERT_HISTORY
WHERE  SCHEMA_NAME = 'DISTRIBUTION'
  AND  SCHEDULED_TIME >= DATEADD('hour', -6, CURRENT_TIMESTAMP())
ORDER  BY SCHEDULED_TIME DESC
LIMIT  20


In [ ]:
-- Demo 3-C: Territory heat map — current snapshot
SELECT territory_name, region, total_aum, advisor_count,
       avg_engagement_score, net_flows_30d, at_risk_advisor_count,
       territory_heat_score
FROM   ANALYTICS_DEV_DB.DISTRIBUTION.TERRITORY_HEAT_MAP
WHERE  as_of_date = (SELECT MAX(as_of_date)
                     FROM ANALYTICS_DEV_DB.DISTRIBUTION.TERRITORY_HEAT_MAP)
ORDER  BY territory_heat_score DESC


---
## Section 4: Self-Service Analytics via Cortex Analyst
The semantic view `DISTRIBUTION_INSIGHTS_SV` defines advisor, territory, fund, and pipeline
business concepts. Wholesalers ask questions in plain English — Cortex Analyst returns
verified SQL and executes it instantly.

Verified queries (VQRs) are matched first for trusted, pre-approved answers.

In [ ]:
# Demo 4: Cortex Analyst via _snowflake.send_snow_api_request()
# _snowflake is the native Snowflake module available in Notebooks and SiS
import _snowflake  # available in Snowflake Notebooks and Streamlit in Snowflake


def ask_analyst(question, sv='ANALYTICS_DEV_DB.DISTRIBUTION.DISTRIBUTION_INSIGHTS_SV'):
    payload = {
        'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': question}]}],
        'semantic_view': sv,
    }
    response = _snowflake.send_snow_api_request(
        'POST',
        '/api/v2/cortex/analyst/message',
        {},    # headers
        {},    # query params
        payload,
        None,  # request_uuid
        30000  # timeout_ms
    )
    if response['status'] != 200:
        return {'error': response.get('content', 'unknown error')}
    return json.loads(response['content'])


questions = [
    'Who are the top 5 advisors by AUM?',
    'Which advisors are at risk of attrition by territory?',
    'Show me the open pipeline value by territory',
    'Give me a summary of our Platinum tier advisors',
    'Which territories have the highest net flows this month?',
]

results = []
for q in questions:
    resp = ask_analyst(q)
    if 'error' in resp:
        print(f'ERROR [{q[:50]}]: {resp["error"]}')
        continue
    sql_stmt, vqr_name, ai_text = None, None, None
    for msg in resp.get('message', {}).get('content', []):
        if msg['type'] == 'text':  ai_text = msg['text']
        if msg['type'] == 'sql':
            sql_stmt = msg['statement']
            v = msg.get('confidence', {}).get('verified_query_used')
            vqr_name = v['name'] if v else None
    results.append({'question': q, 'vqr': vqr_name or 'AI-GEN',
                    'sql_preview': (sql_stmt or '')[:120]})
    label = f'VQR:{vqr_name}' if vqr_name else 'AI-GEN'
    print(f'[{label}] {q}')
    if ai_text:  print(f'  → {ai_text[:150]}')
    print()

print('\nSummary:')
display(pd.DataFrame(results))  # noqa


---
## Deployment Notes

### Upload to Snowflake Stage and create the notebook
```bash
# 1. Upload this file to stage
snow stage copy notebooks/05_snowflake_native_demo.ipynb \
    @ANALYTICS_DEV_DB.DISTRIBUTION.AGENT_STAGE/ --connection your_connection

# 2. Create the Snowflake Notebook object
snow sql --query "
  CREATE OR REPLACE NOTEBOOK ANALYTICS_DEV_DB.DISTRIBUTION.DISTRIBUTION_INSIGHTS_DEMO
  FROM '@ANALYTICS_DEV_DB.DISTRIBUTION.AGENT_STAGE'
  MAIN_FILE = '05_snowflake_native_demo.ipynb'
  QUERY_WAREHOUSE = INGEST;
  ALTER NOTEBOOK ANALYTICS_DEV_DB.DISTRIBUTION.DISTRIBUTION_INSIGHTS_DEMO ADD LIVE VERSION FROM LAST;
" --connection your_connection
```

### Open in Snowsight
Snowsight → Projects → Notebooks → `DISTRIBUTION_INSIGHTS_DEMO`

### Cell bindings (SQL → Python)
Snowflake Notebooks auto-bind SQL cell results as Python variables using the cell `name`.
For example, `s1_signals` (SQL cell) is available as `s1_signals.to_pandas()` in subsequent Python cells.
If running locally in Jupyter, replace `s1_signals` with your own `session.sql(...).to_pandas()`.